# balance Quickstart: New API (SampleFrame / BalanceFrame)

This tutorial demonstrates the **new** `SampleFrame` + `BalanceFrame` API
introduced in balance 0.18.0. It mirrors the original
`balance_quickstart.ipynb` step-by-step, but uses **only** the new classes —
no `Sample` object is needed.

### Why a new API?

| Old API (`Sample`) | New API (`SampleFrame` / `BalanceFrame`) |
|---|---|
| Column roles inferred by exclusion | Column roles declared explicitly |
| Mutable `.set_target()` / `.adjust()` | Immutable — `adjust()` returns a **new** object |
| One class does everything | Clear separation: data container vs. adjustment orchestrator |
| Weight provenance not tracked | Weight metadata recorded per-column |

The old `Sample` API still works and is fully supported; this notebook
simply shows how to do the same analysis with the new classes.

## Analysis

There are four main steps to analysis with the new API:
1. Load data into pandas DataFrames
2. Create `SampleFrame` objects with explicit column roles
3. Build a `BalanceFrame`, adjust, and inspect diagnostics
4. Output results (CSV, download)

## Example dataset

The following is a toy simulated dataset (same data used in the original quickstart).

In [ ]:
%matplotlib inline

import warnings
warnings.filterwarnings("ignore")

from balance import load_data

target_df, sample_df = load_data()

print("target_df: \n", target_df.head())
print("sample_df: \n", sample_df.head())

In practice, you can use `pandas.read_csv()` (or any pandas loader) to
import your own data. The new API also provides `SampleFrame.from_csv()`
for a one-step shortcut.

## Load data into SampleFrame objects

With the old API you would call `Sample.from_frame(df)`. The new API
uses `SampleFrame.from_frame()` where you **explicitly** declare which
columns are covariates, outcomes, etc.  If you omit these arguments,
the factory auto-detects roles the same way `Sample` does (by exclusion
from the id and weight columns).

In [ ]:
from balance import SampleFrame, BalanceFrame

In [ ]:
sample_sf = SampleFrame.from_frame(
    sample_df,
    outcome_columns=["happiness"],
)
# Often times we don't have the outcome for the target.
# In this case we've added it just to validate later that
# the weights indeed help us reduce the bias.
target_sf = SampleFrame.from_frame(
    target_df,
    outcome_columns=["happiness"],
)

### Inspecting a SampleFrame

You can inspect the column roles and data shape at any time.
Unlike `Sample.df`, each role is a separate property — no magic
"everything-that's-left" inference.

In [ ]:
print(f"Covariates:  {sample_sf.covars_columns}")
print(f"Outcomes:    {sample_sf.outcome_columns}")
print(f"Weight cols: {sample_sf.weight_columns}")
print(f"Active wt:   {sample_sf.active_weight_column}")
print(f"ID column:   {sample_sf.id_column_name}")
print(f"Rows:        {len(sample_sf)}")

In [ ]:
sample_sf.df.info()

In [ ]:
print(sample_sf)
print(target_sf)

## Create a BalanceFrame

With the old API you would call `sample.set_target(target)`. The new
API constructs a `BalanceFrame` directly from two `SampleFrame` objects.

A `BalanceFrame` is **immutable** — `adjust()` returns a *new*
BalanceFrame rather than mutating the existing one.

In [ ]:
bf = BalanceFrame(responders=sample_sf, target=target_sf)
print(bf)

## Pre-Adjustment Diagnostics

The `.covars()`, `.weights()`, and `.outcomes()` methods return the
same `BalanceDFCovars` / `BalanceDFWeights` / `BalanceDFOutcomes`
objects as the old API. All of `.mean()`, `.asmd()`, `.plot()`, etc.
work identically.

In [ ]:
print(bf.covars().mean().T)

In [ ]:
print(bf.covars().asmd().T)

In [ ]:
print(bf.covars().asmd(aggregate_by_main_covar=True).T)

### Visualizing the unadjusted comparison

In [ ]:
bf.covars().plot()

## Adjusting Sample to Population

The default method is `'ipw'` (inverse probability/propensity weights
via logistic regression with lasso regularization).

**Key difference from the old API:** `adjust()` returns a **new**
`BalanceFrame` — the original `bf` is unchanged.

In [ ]:
adjusted = bf.adjust()
print(adjusted)

In [ ]:
# The original is still unadjusted:
print(f"bf.is_adjusted        = {bf.is_adjusted}")
print(f"adjusted.is_adjusted  = {adjusted.is_adjusted}")

## Evaluation of the Results

In [ ]:
print(adjusted.summary())

In [ ]:
print(adjusted.covars().mean().T)

We see an improvement in the average ASMD. Detailed per-variable ASMD:

In [ ]:
print(adjusted.covars().asmd().T)

### Covariate plots

In [ ]:
adjusted.covars().plot()

In [ ]:
# Seaborn KDE density plots
adjusted.covars().plot(library="seaborn", dist_type="kde")

### ASCII plots

Use `library="balance"` for a text-based comparison of unadjusted,
adjusted, and target — useful in terminals or logging contexts.

In [ ]:
adjusted.covars().plot(library="balance", bar_width=30);

### Understanding the weights

In [ ]:
adjusted.weights().plot()

In [ ]:
print(adjusted.weights().summary().round(2))

### Design effect and effective sample size

The new API exposes `design_effect()` and `design_effect_prop()`
directly on BalanceFrame — no need to go through `weights().design_effect()`.

In [ ]:
print(f"Design effect:           {adjusted.design_effect():.4f}")
print(f"Effective sample size %: {adjusted.design_effect_prop():.2%}")

## Outcome analysis

In [ ]:
print(adjusted.outcomes().summary())

In [ ]:
adjusted.outcomes().plot()

### Analytics shortcuts

BalanceFrame provides convenience methods for common analytics:

In [ ]:
print("Covariate means (unadjusted / adjusted / target):")
print(adjusted.covar_means())

In [ ]:
print("Outcome SD proportional change:")
print(adjusted.outcome_sd_prop())
print()
print("Outcome variance ratio (adjusted / unadjusted):")
print(adjusted.outcome_variance_ratio())

## Other Adjustment Methods

BalanceFrame supports all the same methods as Sample:
- `"ipw"` — inverse propensity weighting (default)
- `"cbps"` — covariate balancing propensity score
- `"rake"` — iterative proportional fitting (raking)
- `"poststratify"` — post-stratification

Each returns a **new** BalanceFrame — the original stays unchanged.

In [ ]:
adjusted_cbps = bf.adjust(method="cbps")
print(f"CBPS design effect: {adjusted_cbps.design_effect():.4f}")
print(adjusted_cbps.covars().asmd(aggregate_by_main_covar=True).T)

## Transformations

Custom per-column transformations can be set on a `SampleFrame` before
creating the `BalanceFrame`. If no transformations are set, balance
applies its default transformations automatically during `adjust()`.

In [ ]:
# Set default transformations explicitly
sample_sf_t = SampleFrame.from_frame(sample_df, outcome_columns=["happiness"])
sample_sf_t.set_default_transformations()

print("Transformed covariates columns:")
print(sample_sf_t.df_covars_transformed.columns.tolist())

## Diagnostics

`diagnostics()` returns a DataFrame with bias metrics per covariate.

In [ ]:
print(adjusted.diagnostics().to_string())

## Exporting results

The `.df` property combines responder, target, and (if adjusted)
unadjusted data into a single DataFrame with a `source` column.

In [ ]:
combined = adjusted.df
print(combined["source"].value_counts())
print(combined.head())

In [ ]:
# Export to CSV (showing first 500 characters)
print(adjusted.to_csv()[:500])

In [ ]:
adjusted.to_download()

## Filtering rows/columns

`keep_only_some_rows_columns()` returns a new BalanceFrame with
filtered data — the original remains unchanged (immutable pattern).

In [ ]:
filtered = adjusted.keep_only_some_rows_columns(
    row_condition="gender == 'Female'",
    column_names=["gender", "age", "income"],
)
print(f"Original rows: {len(adjusted.responders)}")
print(f"Filtered rows: {len(filtered.responders)}")
print(filtered.df.head())

## Summary: Old vs New API side-by-side

| Step | Old API (`Sample`) | New API (`SampleFrame` / `BalanceFrame`) |
|---|---|---|
| Load data | `s = Sample.from_frame(df)` | `sf = SampleFrame.from_frame(df)` |
| Pair sample + target | `s.set_target(target)` | `bf = BalanceFrame(responders=sf, target=tf)` |
| Adjust | `adjusted = s.adjust()` *(mutates s)* | `adjusted = bf.adjust()` *(bf unchanged)* |
| Summary | `adjusted.summary()` | `adjusted.summary()` |
| Diagnostics | `adjusted.diagnostics()` | `adjusted.diagnostics()` |
| Covariates | `adjusted.covars().mean()` | `adjusted.covars().mean()` |
| Design effect | `adjusted.design_effect()` | `adjusted.design_effect()` |
| CSV export | `adjusted.to_csv()` | `adjusted.to_csv()` |
| Filter | *(not available)* | `adjusted.keep_only_some_rows_columns(...)` |